<a href="https://colab.research.google.com/github/Asritha0507/ML-Market-Basket-Analysis/blob/main/11_Hybrid_Ranking_%26_Ablation_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 11 — Hybrid Ranking & Ablation Study

## Combining Temporal XGBoost Ranking with Product Association Signals

This notebook evaluates whether product-association information can
improve the temporal XGBoost next-basket ranking model.

The temporal XGBoost model provides the primary ranking score using
historical user, product, and temporal features.

Association information is then incorporated as an additional
ranking signal based on products previously purchased together.

### Objectives

- Load the trained temporal XGBoost model and validation candidates.
- Load the association information generated in Notebook 10.
- Calculate an association score for candidate products.
- Compare XGBoost-only and association-enhanced ranking.
- Perform an ablation study using different association sources.
- Evaluate Precision@5, Precision@10, Precision@20.
- Evaluate Recall@5, Recall@10, Recall@20.
- Evaluate NDCG@5, NDCG@10, NDCG@20.
- Identify whether association information improves next-basket ranking.

### Experimental Variants

1. XGBoost-only baseline
2. XGBoost + Apriori association
3. XGBoost + FP-Growth association
4. Hybrid association-enhanced ranking

In [1]:
# ============================================================
# CELL 2 — IMPORT REQUIRED LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import os
import gc

from xgboost import XGBRanker

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================================
# CELL 3 — MOUNT GOOGLE DRIVE AND DEFINE PATHS
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/ML_Market_Basket_Analysis"

DATA_PATH = os.path.join(BASE_PATH, "Datasets")
MODEL_PATH = os.path.join(BASE_PATH, "Models", "Model_Outputs")

ENGINEERED_PATH = os.path.join(
    DATA_PATH,
    "Engineered_Features"
)

MODEL_FILE = os.path.join(
    MODEL_PATH,
    "xgboost_next_basket_ranker.json"
)

ASSOCIATION_FILE = os.path.join(
    MODEL_PATH,
    "association_pairs_fpgrowth.csv"
)

SUMMARY_FILE = os.path.join(
    MODEL_PATH,
    "association_mining_summary.csv"
)

print("Google Drive mounted successfully.")
print("Project paths defined successfully.")
print("Model file:", MODEL_FILE)
print("Association file:", ASSOCIATION_FILE)

Mounted at /content/drive
Google Drive mounted successfully.
Project paths defined successfully.
Model file: /content/drive/MyDrive/ML_Market_Basket_Analysis/Models/Model_Outputs/xgboost_next_basket_ranker.json
Association file: /content/drive/MyDrive/ML_Market_Basket_Analysis/Models/Model_Outputs/association_pairs_fpgrowth.csv


In [3]:
# ============================================================
# CELL 4 — LOAD TRAINED XGBOOST RANKER
# ============================================================

model = XGBRanker()

model.load_model(MODEL_FILE)

print("XGBoost ranking model loaded successfully.")
print("Model file:", MODEL_FILE)

XGBoost ranking model loaded successfully.
Model file: /content/drive/MyDrive/ML_Market_Basket_Analysis/Models/Model_Outputs/xgboost_next_basket_ranker.json


In [4]:
# ============================================================
# CELL 5 — LOAD ASSOCIATION DATA
# ============================================================

association_df = pd.read_csv(
    ASSOCIATION_FILE
)

print("Association data loaded successfully.")
print("Shape:", association_df.shape)

print("\nColumns:")
print(association_df.columns.tolist())

print("\nSample:")
print(
    association_df.head(10).to_string(index=False)
)

Association data loaded successfully.
Shape: (4180, 3)

Columns:
['product_a', 'product_b', 'support']

Sample:
 product_a  product_b  support
     24852      47766 0.016517
     47766      21903 0.009963
     13176      47766 0.007865
     21137      47766 0.008442
     24852      29487 0.004405
     47766      29487 0.002255
     26209      29487 0.001940
     13176      29487 0.001416
     21137      29487 0.001258
     31717      29487 0.001049


In [5]:
# ============================================================
# CELL 6 — LOAD VALIDATION ENGINEERED FEATURES
# ============================================================

validation_chunks = []

for filename in sorted(os.listdir(ENGINEERED_PATH)):
    if filename.endswith(".parquet"):
        file_path = os.path.join(
            ENGINEERED_PATH,
            filename
        )

        chunk = pd.read_parquet(file_path)

        validation_chunk = chunk[
            chunk["order_number"] >= 51
        ].copy()

        if len(validation_chunk) > 0:
            validation_chunks.append(validation_chunk)

        del chunk

validation_data = pd.concat(
    validation_chunks,
    ignore_index=True
)

print("Validation candidate data loaded.")
print("Shape:", validation_data.shape)
print("Unique target orders:",
      validation_data["order_id"].nunique())
print("Positive rows:",
      validation_data["in_next_basket"].sum())

Validation candidate data loaded.
Shape: (2229587, 21)
Unique target orders: 11677
Positive rows: 106926


In [6]:
# ============================================================
# CELL 7 — DEFINE XGBOOST FEATURE COLUMNS
# ============================================================

feature_cols = [
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "times_purchased_before",
    "first_purchase_before",
    "last_purchase_before",
    "recency_before",
    "purchase_span_before",
    "was_previously_purchased",
    "aisle_id",
    "department_id",
    "previous_orders",
    "avg_basket_before",
    "max_basket_before",
    "total_items_before",
    "order_progress"
]

print("Number of model features:", len(feature_cols))
print("\nFeature columns:")
print(feature_cols)

Number of model features: 17

Feature columns:
['order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'times_purchased_before', 'first_purchase_before', 'last_purchase_before', 'recency_before', 'purchase_span_before', 'was_previously_purchased', 'aisle_id', 'department_id', 'previous_orders', 'avg_basket_before', 'max_basket_before', 'total_items_before', 'order_progress']


In [7]:
# ============================================================
# CELL 8 — GENERATE XGBOOST SCORES
# ============================================================

X_validation = validation_data[feature_cols]

validation_data["xgb_score"] = model.predict(
    X_validation
)

print("XGBoost scoring completed.")

print("Score range:")
print(
    "Minimum:", validation_data["xgb_score"].min()
)
print(
    "Maximum:", validation_data["xgb_score"].max()
)

print("\nSample scores:")
print(
    validation_data[
        ["order_id", "product_id", "in_next_basket", "xgb_score"]
    ].head(10).to_string(index=False)
)

del X_validation
gc.collect()

XGBoost scoring completed.
Score range:
Minimum: -5.357573986053467
Maximum: 5.322305202484131

Sample scores:
 order_id  product_id  in_next_basket  xgb_score
  1573122       36929               1   1.300960
  1573122       18523               1   0.968880
  1573122       37067               1  -3.356293
  1573122       24954               1  -1.339180
  1573122       45446               1   0.521998
  1573122       14766               1   1.202665
  1573122       14917               1   0.320249
  1573122       21137               1   2.044389
  1573122        4920               1   1.462731
  1573122        4957               1   2.372794


25

In [8]:
# ============================================================
# CELL 9 — BUILD ASSOCIATION LOOKUP
# ============================================================

association_lookup = {}

for row in association_df.itertuples(index=False):
    key = (int(row.product_a), int(row.product_b))
    association_lookup[key] = float(row.support)

print("Association lookup created successfully.")
print("Number of associations:", len(association_lookup))

# Example lookup
sample_key = next(iter(association_lookup))

print("\nExample association:")
print("Historical product:", sample_key[0])
print("Candidate product:", sample_key[1])
print("Support:", association_lookup[sample_key])

Association lookup created successfully.
Number of associations: 4180

Example association:
Historical product: 24852
Candidate product: 47766
Support: 0.0165172251061821


In [9]:
# ============================================================
# CELL 10 — PREPARE VALIDATION TARGET ORDERS
# ============================================================

target_orders = (
    validation_data[
        ["order_id", "user_id", "order_number"]
    ]
    .drop_duplicates()
    .sort_values("order_id")
    .reset_index(drop=True)
)

print("Validation target orders prepared.")
print("Number of target orders:", len(target_orders))
print("Number of users:", target_orders["user_id"].nunique())

print("\nOrder-number range:")
print(
    target_orders["order_number"].min(),
    "to",
    target_orders["order_number"].max()
)

print("\nSample:")
print(
    target_orders.head(10).to_string(index=False)
)

Validation target orders prepared.
Number of target orders: 11677
Number of users: 10238

Order-number range:
51 to 99

Sample:
 order_id  user_id  order_number
      240   183623            64
      339    34416            57
      623    37804            63
      654    54729            52
      798   111550            75
     1033   192444            54
     1208   194615            51
     1309   115961            55
     1349   134597            53
     1692   137276            79


In [10]:
# ============================================================
# CELL 11 — LOAD HISTORICAL ORDER INFORMATION
# ============================================================

orders = pd.read_csv(
    os.path.join(DATA_PATH, "orders.csv")
)

historical_orders = orders.merge(
    target_orders[["user_id", "order_number"]],
    on="user_id",
    suffixes=("_history", "_target")
)

historical_orders = historical_orders[
    historical_orders["order_number_history"]
    < historical_orders["order_number_target"]
][
    ["order_id", "user_id", "order_number_history"]
].copy()

print("Historical orders identified successfully.")
print("Historical order rows:", len(historical_orders))
print("Users covered:",
      historical_orders["user_id"].nunique())

print("\nSample:")
print(
    historical_orders.head(10).to_string(index=False)
)

del orders
gc.collect()

Historical orders identified successfully.
Historical order rows: 778917
Users covered: 10238

Sample:
 order_id  user_id  order_number_history
   361493       27                     1
  1662354       27                     2
   965677       27                     3
  2842504       27                     4
  1007361       27                     5
  2792524       27                     6
  1771198       27                     7
  1460681       27                     8
  3317805       27                     9
  3149167       27                    10


0

In [11]:
# ============================================================
# CELL 12 — LOAD HISTORICAL PRODUCT PURCHASES
# ============================================================

PRIOR_FILE = os.path.join(
    DATA_PATH,
    "order_products_prior.csv"
)

historical_order_ids = set(
    historical_orders["order_id"].astype(int)
)

historical_products = []

for chunk in pd.read_csv(
    PRIOR_FILE,
    usecols=["order_id", "product_id"],
    chunksize=500000
):
    filtered = chunk[
        chunk["order_id"].isin(historical_order_ids)
    ]

    if len(filtered) > 0:
        historical_products.append(
            filtered
        )

    del chunk

historical_products = pd.concat(
    historical_products,
    ignore_index=True
)

print("Historical product purchases loaded.")
print("Rows:", len(historical_products))
print(
    "Unique historical orders:",
    historical_products["order_id"].nunique()
)
print(
    "Unique products:",
    historical_products["product_id"].nunique()
)

print("\nSample:")
print(
    historical_products.head(10).to_string(index=False)
)

gc.collect()

Historical product purchases loaded.
Rows: 6893889
Unique historical orders: 705623
Unique products: 41334

Sample:
 order_id  product_id
        4       46842
        4       26434
        4       39758
        4       27761
        4       10054
        4       21351
        4       22598
        4       34862
        4       40285
        4       17616


0

In [12]:
# ============================================================
# CELL 13 — BUILD TARGET ORDER HISTORY
# ============================================================

history_map = (
    historical_orders[
        ["order_id", "user_id", "order_number_history"]
    ]
    .merge(
        historical_products,
        on="order_id",
        how="inner"
    )
)

target_history = history_map.merge(
    target_orders[
        ["order_id", "user_id", "order_number"]
    ],
    on="user_id",
    how="inner",
    suffixes=("_history", "_target")
)

target_history = target_history[
    target_history["order_number_history"]
    < target_history["order_number"]
][
    ["order_id_target", "product_id"]
].drop_duplicates()

target_history = target_history.rename(
    columns={"order_id_target": "target_order_id"}
)

print("Target history mapping created.")
print("Rows:", len(target_history))
print(
    "Target orders covered:",
    target_history["target_order_id"].nunique()
)
print(
    "Unique historical products:",
    target_history["product_id"].nunique()
)

print("\nSample:")
print(
    target_history.head(10).to_string(index=False)
)

del history_map
gc.collect()

Target history mapping created.
Rows: 1907797
Target orders covered: 11677
Unique historical products: 41334

Sample:
 target_order_id  product_id
         1573122       25718
         1573122       30776
         1573122        9604
         1573122        6287
         1573122       20947
         1573122       48559
         1573122       36929
         1573122       16768
         1573122        1194
         1573122       19503


10

In [13]:
# ============================================================
# CELL 14 — CALCULATE ASSOCIATION SCORES
# ============================================================

# Create a mapping:
# target_order_id -> set of historically purchased products

history_sets = (
    target_history
    .groupby("target_order_id")["product_id"]
    .apply(set)
    .to_dict()
)

print("History sets created.")
print("Target orders with history:", len(history_sets))

# Calculate maximum association support
def get_association_score(target_order_id, candidate_product):
    history = history_sets.get(target_order_id, set())

    max_support = 0.0

    for historical_product in history:
        support = association_lookup.get(
            (historical_product, candidate_product),
            0.0
        )

        if support > max_support:
            max_support = support

    return max_support

association_scores = []

for row in validation_data[
    ["order_id", "product_id"]
].itertuples(index=False):

    score = get_association_score(
        row.order_id,
        row.product_id
    )

    association_scores.append(score)

validation_data["association_score"] = np.array(
    association_scores,
    dtype=np.float32
)

print("\nAssociation scores calculated.")

print(
    "Non-zero association scores:",
    (validation_data["association_score"] > 0).sum()
)

print(
    "Maximum association score:",
    validation_data["association_score"].max()
)

print("\nSample:")
print(
    validation_data[
        [
            "order_id",
            "product_id",
            "xgb_score",
            "association_score"
        ]
    ].head(10).to_string(index=False)
)

History sets created.
Target orders with history: 11677

Association scores calculated.
Non-zero association scores: 972515
Maximum association score: 0.02107912488281727

Sample:
 order_id  product_id  xgb_score  association_score
  1573122       36929   1.300960           0.000000
  1573122       18523   0.968880           0.002569
  1573122       37067  -3.356293           0.000000
  1573122       24954  -1.339180           0.000000
  1573122       45446   0.521998           0.000000
  1573122       14766   1.202665           0.000000
  1573122       14917   0.320249           0.000000
  1573122       21137   2.044389           0.021079
  1573122        4920   1.462731           0.007760
  1573122        4957   2.372794           0.003723


In [14]:
# ============================================================
# CELL 15 — NORMALIZE RANKING SIGNALS
# ============================================================

def min_max_normalize(series):
    min_value = series.min()
    max_value = series.max()

    if max_value == min_value:
        return pd.Series(
            np.zeros(len(series)),
            index=series.index
        )

    return (series - min_value) / (max_value - min_value)


validation_data["xgb_normalized"] = (
    validation_data
    .groupby("order_id")["xgb_score"]
    .transform(min_max_normalize)
)

validation_data["association_normalized"] = (
    validation_data
    .groupby("order_id")["association_score"]
    .transform(min_max_normalize)
)

print("Ranking signals normalized successfully.")

print("\nNormalized XGBoost range:")
print(
    validation_data["xgb_normalized"].min(),
    "to",
    validation_data["xgb_normalized"].max()
)

print("\nNormalized association range:")
print(
    validation_data["association_normalized"].min(),
    "to",
    validation_data["association_normalized"].max()
)

print("\nSample:")
print(
    validation_data[
        [
            "order_id",
            "product_id",
            "xgb_normalized",
            "association_normalized"
        ]
    ].head(10).to_string(index=False)
)

Ranking signals normalized successfully.

Normalized XGBoost range:
0.0 to 1.0

Normalized association range:
0.0 to 1.0

Sample:
 order_id  product_id  xgb_normalized  association_normalized
  1573122       36929        0.837239                0.000000
  1573122       18523        0.788110                0.121891
  1573122       37067        0.148231                0.000000
  1573122       24954        0.446649                0.000000
  1573122       45446        0.721997                0.000000
  1573122       14766        0.822697                0.000000
  1573122       14917        0.692150                0.000000
  1573122       21137        0.947224                1.000000
  1573122        4920        0.861172                0.368159
  1573122        4957        0.995810                0.176617


In [15]:
# ============================================================
# CELL 16 — CALCULATE HYBRID SCORE
# ============================================================

ASSOCIATION_WEIGHT = 0.20
XGB_WEIGHT = 1 - ASSOCIATION_WEIGHT

validation_data["hybrid_score"] = (
    XGB_WEIGHT * validation_data["xgb_normalized"]
    + ASSOCIATION_WEIGHT * validation_data["association_normalized"]
)

print("Hybrid ranking score calculated.")

print("XGBoost weight:", XGB_WEIGHT)
print("Association weight:", ASSOCIATION_WEIGHT)

print("\nHybrid score range:")
print(
    validation_data["hybrid_score"].min(),
    "to",
    validation_data["hybrid_score"].max()
)

print("\nSample:")
print(
    validation_data[
        [
            "order_id",
            "product_id",
            "xgb_normalized",
            "association_normalized",
            "hybrid_score"
        ]
    ].head(10).to_string(index=False)
)

Hybrid ranking score calculated.
XGBoost weight: 0.8
Association weight: 0.2

Hybrid score range:
0.0 to 1.0

Sample:
 order_id  product_id  xgb_normalized  association_normalized  hybrid_score
  1573122       36929        0.837239                0.000000      0.669791
  1573122       18523        0.788110                0.121891      0.654866
  1573122       37067        0.148231                0.000000      0.118585
  1573122       24954        0.446649                0.000000      0.357319
  1573122       45446        0.721997                0.000000      0.577598
  1573122       14766        0.822697                0.000000      0.658158
  1573122       14917        0.692150                0.000000      0.553720
  1573122       21137        0.947224                1.000000      0.957780
  1573122        4920        0.861172                0.368159      0.762570
  1573122        4957        0.995810                0.176617      0.831971


In [16]:
# ============================================================
# CELL 17 — DEFINE RANKING EVALUATION FUNCTION
# ============================================================

def dcg_at_k(relevances, k):
    relevances = np.asarray(relevances[:k])

    if len(relevances) == 0:
        return 0.0

    discounts = np.log2(
        np.arange(2, len(relevances) + 2)
    )

    return np.sum(
        relevances / discounts
    )


def evaluate_ranking(data, score_column, k):
    precisions = []
    recalls = []
    ndcgs = []

    for _, group in data.groupby("order_id"):

        ranked = group.sort_values(
            score_column,
            ascending=False
        )

        top_k = ranked.head(k)

        relevance = top_k[
            "in_next_basket"
        ].values

        hits = relevance.sum()

        actual_products = group[
            "in_next_basket"
        ].sum()

        precision = hits / k

        recall = (
            hits / actual_products
            if actual_products > 0
            else 0
        )

        ideal_relevance = np.ones(
            min(int(actual_products), k)
        )

        ideal_dcg = dcg_at_k(
            ideal_relevance,
            k
        )

        ndcg = (
            dcg_at_k(relevance, k) / ideal_dcg
            if ideal_dcg > 0
            else 0
        )

        precisions.append(precision)
        recalls.append(recall)
        ndcgs.append(ndcg)

    return {
        "Precision": np.mean(precisions),
        "Recall": np.mean(recalls),
        "NDCG": np.mean(ndcgs)
    }


print("Ranking evaluation function defined successfully.")

Ranking evaluation function defined successfully.


In [17]:
# ============================================================
# CELL 18 — EVALUATE XGBOOST-ONLY BASELINE
# ============================================================

xgb_results = {}

for k in [5, 10, 20]:

    metrics = evaluate_ranking(
        validation_data,
        "xgb_score",
        k
    )

    xgb_results[k] = metrics

    print(
        f"XGBoost @ {k}: "
        f"Precision={metrics['Precision']:.4f}, "
        f"Recall={metrics['Recall']:.4f}, "
        f"NDCG={metrics['NDCG']:.4f}"
    )

XGBoost @ 5: Precision=0.5951, Recall=0.4682, NDCG=0.7046
XGBoost @ 10: Precision=0.4576, Recall=0.6414, NDCG=0.7023
XGBoost @ 20: Precision=0.3140, Recall=0.7930, NDCG=0.7384


In [18]:
# ============================================================
# CELL 20 — EVALUATE HYBRID RANKING
# ============================================================

hybrid_results = {}

for k in [5, 10, 20]:

    metrics = evaluate_ranking(
        validation_data,
        "hybrid_score",
        k
    )

    hybrid_results[k] = metrics

    print(
        f"Hybrid @ {k}: "
        f"Precision={metrics['Precision']:.4f}, "
        f"Recall={metrics['Recall']:.4f}, "
        f"NDCG={metrics['NDCG']:.4f}"
    )

Hybrid @ 5: Precision=0.5849, Recall=0.4625, NDCG=0.6909
Hybrid @ 10: Precision=0.4544, Recall=0.6388, NDCG=0.6934
Hybrid @ 20: Precision=0.3132, Recall=0.7937, NDCG=0.7319


In [19]:
# ============================================================
# CELL 21 — TEST ASSOCIATION WEIGHTS
# ============================================================

weights = [0.05, 0.10, 0.15]

weight_results = {}

for association_weight in weights:

    xgb_weight = 1 - association_weight

    validation_data["test_hybrid_score"] = (
        xgb_weight * validation_data["xgb_normalized"]
        + association_weight * validation_data["association_normalized"]
    )

    weight_results[association_weight] = {}

    print(f"\nAssociation weight = {association_weight:.2f}")

    for k in [5, 10, 20]:

        metrics = evaluate_ranking(
            validation_data,
            "test_hybrid_score",
            k
        )

        weight_results[association_weight][k] = metrics

        print(
            f"@{k}: "
            f"P={metrics['Precision']:.4f}, "
            f"R={metrics['Recall']:.4f}, "
            f"NDCG={metrics['NDCG']:.4f}"
        )


Association weight = 0.05
@5: P=0.5955, R=0.4685, NDCG=0.7052
@10: P=0.4580, R=0.6419, NDCG=0.7028
@20: P=0.3144, R=0.7942, NDCG=0.7392

Association weight = 0.10
@5: P=0.5942, R=0.4675, NDCG=0.7036
@10: P=0.4580, R=0.6419, NDCG=0.7022
@20: P=0.3145, R=0.7949, NDCG=0.7389

Association weight = 0.15
@5: P=0.5909, R=0.4660, NDCG=0.6990
@10: P=0.4569, R=0.6409, NDCG=0.6991
@20: P=0.3141, R=0.7948, NDCG=0.7365


In [20]:
# ============================================================
# CELL 22 — CREATE ABLATION RESULTS TABLE
# ============================================================

ablation_rows = []

# XGBoost-only baseline
for k in [5, 10, 20]:

    ablation_rows.append({
        "Variant": "XGBoost-only",
        "Association_Weight": 0.00,
        "K": k,
        "Precision": xgb_results[k]["Precision"],
        "Recall": xgb_results[k]["Recall"],
        "NDCG": xgb_results[k]["NDCG"]
    })

# Association-enhanced variants
for weight, results in weight_results.items():

    for k in [5, 10, 20]:

        ablation_rows.append({
            "Variant": "XGBoost + Association",
            "Association_Weight": weight,
            "K": k,
            "Precision": results[k]["Precision"],
            "Recall": results[k]["Recall"],
            "NDCG": results[k]["NDCG"]
        })

ablation_results = pd.DataFrame(
    ablation_rows
)

print("Ablation results table created.")
print(
    ablation_results.to_string(index=False)
)

Ablation results table created.
              Variant  Association_Weight  K  Precision   Recall     NDCG
         XGBoost-only                0.00  5   0.595067 0.468219 0.704610
         XGBoost-only                0.00 10   0.457635 0.641392 0.702260
         XGBoost-only                0.00 20   0.313955 0.793049 0.738427
XGBoost + Association                0.05  5   0.595495 0.468467 0.705215
XGBoost + Association                0.05 10   0.457977 0.641901 0.702785
XGBoost + Association                0.05 20   0.314370 0.794152 0.739163
XGBoost + Association                0.10  5   0.594159 0.467498 0.703569
XGBoost + Association                0.10 10   0.458020 0.641901 0.702165
XGBoost + Association                0.10 20   0.314503 0.794935 0.738893
XGBoost + Association                0.15  5   0.590888 0.465973 0.699000
XGBoost + Association                0.15 10   0.456915 0.640858 0.699077
XGBoost + Association                0.15 20   0.314147 0.794808 0.736496


In [21]:
# ============================================================
# CELL 23 — CALCULATE ABLATION IMPROVEMENTS
# ============================================================

baseline = ablation_results[
    ablation_results["Variant"] == "XGBoost-only"
][
    ["K", "Precision", "Recall", "NDCG"]
].rename(
    columns={
        "Precision": "Baseline_Precision",
        "Recall": "Baseline_Recall",
        "NDCG": "Baseline_NDCG"
    }
)

hybrid_comparison = ablation_results[
    ablation_results["Variant"] == "XGBoost + Association"
].merge(
    baseline,
    on="K"
)

hybrid_comparison["Precision_Change"] = (
    hybrid_comparison["Precision"]
    - hybrid_comparison["Baseline_Precision"]
)

hybrid_comparison["Recall_Change"] = (
    hybrid_comparison["Recall"]
    - hybrid_comparison["Baseline_Recall"]
)

hybrid_comparison["NDCG_Change"] = (
    hybrid_comparison["NDCG"]
    - hybrid_comparison["Baseline_NDCG"]
)

print("Ablation improvements calculated.")

print(
    hybrid_comparison[
        [
            "Association_Weight",
            "K",
            "Precision_Change",
            "Recall_Change",
            "NDCG_Change"
        ]
    ].to_string(index=False)
)

Ablation improvements calculated.
 Association_Weight  K  Precision_Change  Recall_Change  NDCG_Change
               0.05  5          0.000428       0.000247     0.000606
               0.05 10          0.000343       0.000509     0.000525
               0.05 20          0.000415       0.001103     0.000737
               0.10  5         -0.000908      -0.000721    -0.001041
               0.10 10          0.000385       0.000509    -0.000094
               0.10 20          0.000548       0.001886     0.000467
               0.15  5         -0.004179      -0.002247    -0.005610
               0.15 10         -0.000719      -0.000534    -0.003183
               0.15 20          0.000193       0.001759    -0.001931


In [22]:
# ============================================================
# CELL 24 — SAVE ABLATION RESULTS
# ============================================================

ABLATION_FILE = os.path.join(
    MODEL_PATH,
    "hybrid_ablation_results.csv"
)

IMPROVEMENT_FILE = os.path.join(
    MODEL_PATH,
    "hybrid_ablation_improvements.csv"
)

ablation_results.to_csv(
    ABLATION_FILE,
    index=False
)

hybrid_comparison.to_csv(
    IMPROVEMENT_FILE,
    index=False
)

print("Ablation results saved successfully.")
print("Ablation file:", ABLATION_FILE)
print("Improvement file:", IMPROVEMENT_FILE)

print("\nFiles exist:")
print("Ablation:", os.path.exists(ABLATION_FILE))
print("Improvement:", os.path.exists(IMPROVEMENT_FILE))

Ablation results saved successfully.
Ablation file: /content/drive/MyDrive/ML_Market_Basket_Analysis/Models/Model_Outputs/hybrid_ablation_results.csv
Improvement file: /content/drive/MyDrive/ML_Market_Basket_Analysis/Models/Model_Outputs/hybrid_ablation_improvements.csv

Files exist:
Ablation: True
Improvement: True


In [23]:
# ============================================================
# CELL 25 — FINAL XGBOOST VS HYBRID COMPARISON
# ============================================================

final_comparison = pd.DataFrame({
    "K": [5, 10, 20],

    "XGB_Precision": [
        xgb_results[k]["Precision"]
        for k in [5, 10, 20]
    ],

    "Hybrid_Precision": [
        weight_results[0.05][k]["Precision"]
        for k in [5, 10, 20]
    ],

    "XGB_Recall": [
        xgb_results[k]["Recall"]
        for k in [5, 10, 20]
    ],

    "Hybrid_Recall": [
        weight_results[0.05][k]["Recall"]
        for k in [5, 10, 20]
    ],

    "XGB_NDCG": [
        xgb_results[k]["NDCG"]
        for k in [5, 10, 20]
    ],

    "Hybrid_NDCG": [
        weight_results[0.05][k]["NDCG"]
        for k in [5, 10, 20]
    ]
})

print("Final comparison:")
print(
    final_comparison.to_string(index=False)
)

Final comparison:
 K  XGB_Precision  Hybrid_Precision  XGB_Recall  Hybrid_Recall  XGB_NDCG  Hybrid_NDCG
 5       0.595067          0.595495    0.468219       0.468467  0.704610     0.705215
10       0.457635          0.457977    0.641392       0.641901  0.702260     0.702785
20       0.313955          0.314370    0.793049       0.794152  0.738427     0.739163


In [24]:
# ============================================================
# CELL 26 — SAVE FINAL COMPARISON
# ============================================================

FINAL_COMPARISON_FILE = os.path.join(
    MODEL_PATH,
    "xgboost_vs_hybrid_final_comparison.csv"
)

final_comparison.to_csv(
    FINAL_COMPARISON_FILE,
    index=False
)

print("Final comparison saved successfully.")
print("File:", FINAL_COMPARISON_FILE)
print(
    "File exists:",
    os.path.exists(FINAL_COMPARISON_FILE)
)

Final comparison saved successfully.
File: /content/drive/MyDrive/ML_Market_Basket_Analysis/Models/Model_Outputs/xgboost_vs_hybrid_final_comparison.csv
File exists: True


## Final Interpretation

The temporal XGBoost model provides a strong baseline for next-basket ranking.

Adding the FP-Growth association signal with a low weight of 0.05 produces only marginal changes in performance:

- Precision@5 increases from 0.5951 to 0.5955.
- Recall@10 increases from 0.6414 to 0.6419.
- Recall@20 increases from 0.7930 to 0.7942.
- NDCG@20 increases from 0.7384 to 0.7392.

Higher association weights do not consistently improve the ranking results, and the 0.20 blend reduces performance across the evaluated metrics.

Therefore, association information provides a small complementary signal, but the temporal XGBoost features remain the primary source of ranking performance.

The results do not indicate a large performance improvement from association mining. Instead, they show that association information can be incorporated as a lightweight supporting signal without replacing the temporal ranking model.

In [25]:
# ============================================================
# CELL 28 — VERIFY NOTEBOOK 11 OUTPUTS
# ============================================================

output_files = [
    os.path.join(MODEL_PATH, "association_pairs_fpgrowth.csv"),
    os.path.join(MODEL_PATH, "association_mining_summary.csv"),
    os.path.join(MODEL_PATH, "hybrid_ablation_results.csv"),
    os.path.join(MODEL_PATH, "hybrid_ablation_improvements.csv"),
    os.path.join(MODEL_PATH, "xgboost_vs_hybrid_final_comparison.csv")
]

print("Notebook 11 output verification:\n")

for file_path in output_files:
    print(
        os.path.basename(file_path),
        "->",
        "Exists" if os.path.exists(file_path) else "Missing"
    )

Notebook 11 output verification:

association_pairs_fpgrowth.csv -> Exists
association_mining_summary.csv -> Exists
hybrid_ablation_results.csv -> Exists
hybrid_ablation_improvements.csv -> Exists
xgboost_vs_hybrid_final_comparison.csv -> Exists


In [26]:
# ============================================================
# FINAL VERIFICATION — NOTEBOOK 11
# ============================================================

check_df = pd.read_csv(FINAL_COMPARISON_FILE)

print("Final comparison shape:", check_df.shape)
print("\nK values:", sorted(check_df["K"].unique()))
print("Missing values:", check_df.isnull().sum().sum())

print("\nFinal comparison:")
print(check_df.to_string(index=False))

Final comparison shape: (3, 7)

K values: [np.int64(5), np.int64(10), np.int64(20)]
Missing values: 0

Final comparison:
 K  XGB_Precision  Hybrid_Precision  XGB_Recall  Hybrid_Recall  XGB_NDCG  Hybrid_NDCG
 5       0.595067          0.595495    0.468219       0.468467  0.704610     0.705215
10       0.457635          0.457977    0.641392       0.641901  0.702260     0.702785
20       0.313955          0.314370    0.793049       0.794152  0.738427     0.739163
